# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `SatelliteZonalStats`. openEO creates one cloud-masked, indexed monthly NetCDF cube per spatial batch. IQR cleaning, null filling, and parcel statistics run locally, producing one ML-ready GeoParquet row per parcel with dated feature columns.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Run this notebook from within the satellites repository.")
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from openeo_parcel_stats_pipeline.zonal_stats_job_manager import (
    JobManagerSatelliteZonalStats,
    compute_tile_buffer_metres,
    compute_tile_width,
)
from common_libraries.io_library import read_data
import common_libraries.geom_library as geom_l

## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`acquisition pixel indices`
`monthly pixel median`
`monthly per-variable cube IQR outlier removal`
`local null filling`
`parcel statistics`
`one-row-per-parcel time-series pivot`
`ML-ready GeoParquet`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [ ]:
WORK_PATH = Path("C:/work_dir")
PARCELS_PATH = WORK_PATH / "kozani-2023-2024" / "kozani_parcels.gpkg"

In [ ]:
gdf_parcels = read_data(str(PARCELS_PATH))
gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
print(len(gdf_parcels), "parcels in test area")
gdf_parcels.head(3)

In [ ]:
class_column = "label"
parcel_id_column = "parcel_code"

In [ ]:
START_DATE = "2023-10-01"
END_DATE = "2024-09-30"
RUN_IDENTIFIER = "kozani-2023-2024"  # Change this for each named run.
OUTPUT_DIR = WORK_PATH / RUN_IDENTIFIER / "1_parcel_stats"
WORKING_EPSG = 32634

CALCULATE_SENTINEL2_INDICES = True
SENTINEL2_INDICES = ["EVI", "NDMI", "NDRE", "NDVI", "NDWI",  "SAVI"]
SENTINEL2_BANDS = ["B02", "B03", "B04", "B05", "B08", "B11", "B12"]
SENTINEL1_BANDS = [] # ["VV", "VH"]

# Mean is mandatory; percentiles enable derived IQR and robust skewness.
SPATIAL_STATISTICS = [
    "mean",
    "median",
    "sd",
    "min",
    "max",
    # "count",
    "p10",
    "p25",
    "p75",
    "p90",
]

INTERPOLATION_METHOD = "nearest"  # "idw"
INTERPOLATION_MAX_DISTANCE_IN_METERS = 500 

# IQR outliers become null, then nulls are filled.
REMOVE_OUTLIERS = True
IQR_QUANTILES = (0.1, 0.9)
IQR_MULTIPLIER = 1.5
FILL_NULLS = True
IQR_MIN_VALID_PIXELS = 20
TEMPORAL_FILL_MODE = "bidirectional" # or past_only
MINIMUM_PARCEL_PIXELS = 3
MINIMUM_OBSERVED_FRACTION_FOR_FILL = 0.20

# Each worker loads one complete monthly cube; use one unless RAM has been sized for parallel cubes.
LOCAL_BATCH_WORKERS = 1

gdf_parcels = gdf_parcels.to_crs(WORKING_EPSG)

## Tile-size recommendation

This local comparison selects the smallest candidate grid whose buffered ownership tiles fully cover every parcel.

In [ ]:
TILE_SIZE_METRES = compute_tile_width(gdf_parcels, parcel_id_col=parcel_id_column, working_epsg=WORKING_EPSG)
TILE_BUFFER_METRES = compute_tile_buffer_metres(gdf_parcels, WORKING_EPSG)

## Run extraction

In [ ]:
TILE_SIZE_METRES = 30_000
TILE_BUFFER_METRES = 3_000

In [ ]:
# OPENEO_USERS format: username,password;another_username,another_password
credentials = [
    tuple(part.strip() for part in entry.strip().split(",", maxsplit=1))
    for entry in os.environ["OPENEO_USERS"].split(";")
    if entry.strip()
]
if not credentials or any(len(pair) != 2 or not all(pair) for pair in credentials):
    raise ValueError("OPENEO_USERS must contain one or more username,password pairs.")

username, password = credentials[0]  # First user

In [ ]:
extractor = JobManagerSatelliteZonalStats(
    parcels=gdf_parcels,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
    working_epsg=WORKING_EPSG,
    
    openeo_username=username,
    openeo_password=password,
    openeo_parallel_jobs=2,
    tile_size_metres=TILE_SIZE_METRES,
    tile_buffer_metres=TILE_BUFFER_METRES,
    job_poll_seconds=30,
    run_identifier=RUN_IDENTIFIER,
        
    parcel_id_field=parcel_id_column,
    sentinel2_bands=SENTINEL2_BANDS,
    calculate_sentinel2_indices=CALCULATE_SENTINEL2_INDICES,
    sentinel2_indices=SENTINEL2_INDICES,
    sentinel1_bands=SENTINEL1_BANDS,
    spatial_statistics=SPATIAL_STATISTICS,
    
    minimum_parcel_pixels=MINIMUM_PARCEL_PIXELS,
    minimum_observed_fraction_for_fill=MINIMUM_OBSERVED_FRACTION_FOR_FILL,
        
    temporal_fill_mode=TEMPORAL_FILL_MODE,
    spatial_fill_window_sizes=(3, 5),
    interpolation_method=INTERPOLATION_METHOD,
    interpolation_max_distance_in_meters=INTERPOLATION_MAX_DISTANCE_IN_METERS,
    
    remove_outliers=REMOVE_OUTLIERS,
    iqr_quantiles=IQR_QUANTILES,
    iqr_multiplier=IQR_MULTIPLIER,
    fill_nulls=FILL_NULLS,
    iqr_min_valid_pixels=IQR_MIN_VALID_PIXELS,
    
    batch_workers=LOCAL_BATCH_WORKERS,
)
results = extractor.run()
results.head()